In [ ]:
def configure_optimizer_with_llrd(model, criterion, base_lr=2e-5, head_lr=2e-4, decay_factor=0.9):
    optimizer_grouped_parameters = []
    processed_params = set()

    encoder_layers = model.wav2vec2.encoder.layers
    num_layers = len(encoder_layers)

    base_params = list(model.wav2vec2.feature_extractor.parameters()) + list(model.wav2vec2.feature_projection.parameters())
    optimizer_grouped_parameters.append({"params": base_params, "lr": base_lr * (decay_factor ** num_layers), "weight_decay": 0.0})
    for p in base_params: processed_params.add(id(p))

    for idx, layer in enumerate(encoder_layers):
        layer_lr = base_lr * (decay_factor ** (num_layers - 1 - idx))
        layer_params = list(layer.parameters())
        optimizer_grouped_parameters.append({"params": layer_params, "lr": layer_lr, "weight_decay": 0.01})
        for p in layer_params: processed_params.add(id(p))

    head_params = []
    for p in model.parameters():
        if id(p) not in processed_params and p.requires_grad:
            head_params.append(p)
    for p in criterion.parameters():
        if p.requires_grad:
            head_params.append(p)

    optimizer_grouped_parameters.append({"params": head_params, "lr": head_lr, "weight_decay": 1e-4})
    return torch.optim.AdamW(optimizer_grouped_parameters)

EPOCHS = 20
BATCH_SIZE = 16
GRAD_ACCUM = 4

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

oof_preds = np.zeros(len(train_meta), dtype=np.int32)
fold_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(cv_folds, 1):
    print(f"\n{'='*15} FOLD {fold} {'='*15}")
    sub_train_ds = Subset(train_ds, train_idx)
    sub_val_ds = Subset(train_ds, val_idx)

    checkpoint_path = os.path.join(MODELS_DIR, f"model_fold_{fold}.pt")
    history_path = os.path.join(LOGS_DIR, f"history_fold_{fold}.csv")

    model = SERModel().to(DEVICE)
    criterion = ArcFaceLoss(
        num_classes=4, embedding_size=256,
        margins=[0.45, 0.20, 0.40, 0.45], scale=20.0,
        class_weights=alpha_weights.to(DEVICE), gamma=2.0
    ).to(DEVICE)

    val_loader = DataLoader(
        sub_val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=AudioDataCollator(feature_extractor, augment=False),
        num_workers=0, pin_memory=True
    )

    if os.path.exists(checkpoint_path):
        print(f"[HỆ THỐNG] Đã tìm thấy checkpoint Fold {fold}. Đang load để đánh giá...")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state'])
        criterion.load_state_dict(checkpoint['criterion_state'])
        model.eval()
        criterion.eval()

        val_preds, val_targets = [], []
        with torch.inference_mode():
            for batch in val_loader:
                inputs = batch["input_values"].to(DEVICE, non_blocking=True)
                mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
                targets = batch["labels"].to(DEVICE, non_blocking=True)

                with torch.amp.autocast(device_type=DEVICE.type):
                    embeddings = model(inputs, mask)

                scale_val = criterion.scale.item() if isinstance(criterion.scale, torch.Tensor) else 20.0
                cosine_eval = F.linear(F.normalize(embeddings.float()), F.normalize(criterion.weight.float())) * scale_val
                preds = torch.argmax(cosine_eval, dim=-1).cpu().numpy()
                val_preds.extend(preds)
                val_targets.extend(targets.cpu().numpy())

        best_fold_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        oof_preds[val_idx] = val_preds
        fold_f1_scores.append(best_fold_f1)

        print(f"\n--> BÁO CÁO FOLD {fold} (LOADED FROM DRIVE):")
        print(f"Accuracy : {accuracy_score(val_targets, val_preds):.4f} | F1: {best_fold_f1:.4f}")
        print(classification_report(val_targets, val_preds, target_names=list(LABEL2ID.keys()), zero_division=0))
        plot_confusion_matrix(val_targets, val_preds, fold, best_fold_f1)

        del model, criterion, val_loader
        gc.collect()
        torch.cuda.empty_cache()
        continue

    train_loader = DataLoader(
        sub_train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=AudioDataCollator(feature_extractor, augment=True),
        num_workers=0, pin_memory=True, drop_last=True
    )

    optimizer = configure_optimizer_with_llrd(model, criterion)
    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps = max(200, int(0.1 * total_steps))
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = torch.amp.GradScaler('cuda' if DEVICE.type == 'cuda' else 'cpu')

    best_fold_f1, best_fold_preds, best_fold_targets = 0.0, [], []
    epoch_history = []

    for epoch in range(EPOCHS):
        model.train()
        criterion.train()
        optimizer.zero_grad()
        train_loss = 0.0

        train_pbar = tqdm(train_loader, desc=f"Fold {fold} - Train Ep {epoch+1}/{EPOCHS}", leave=False)
        for step, batch in enumerate(train_pbar):
            inputs = batch["input_values"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            targets = batch["labels"].to(DEVICE, non_blocking=True)

            with torch.amp.autocast(device_type=DEVICE.type):
                embeddings = model(inputs, mask)

            loss = criterion(embeddings.float(), targets) / GRAD_ACCUM

            scaler.scale(loss).backward()
            train_loss += loss.item() * GRAD_ACCUM

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(criterion.parameters()), max_norm=1.0)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()

                if scale_before <= scaler.get_scale():
                    scheduler.step()
                optimizer.zero_grad()

            train_pbar.set_postfix({"loss": f"{loss.item() * GRAD_ACCUM:.4f}"})

        model.eval()
        criterion.eval()
        val_preds, val_targets, val_loss = [], [], 0.0

        val_pbar = tqdm(val_loader, desc=f"Fold {fold} - Eval Ep {epoch+1}/{EPOCHS}", leave=False)
        with torch.inference_mode():
            for batch in val_pbar:
                inputs = batch["input_values"].to(DEVICE, non_blocking=True)
                mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
                targets = batch["labels"].to(DEVICE, non_blocking=True)

                with torch.amp.autocast(device_type=DEVICE.type):
                    embeddings = model(inputs, mask)

                batch_loss = criterion(embeddings.float(), targets)
                scale_val = criterion.scale.item() if isinstance(criterion.scale, torch.Tensor) else 20.0
                cosine_eval = F.linear(F.normalize(embeddings.float()), F.normalize(criterion.weight.float())) * scale_val

                val_loss += batch_loss.item()
                val_preds.extend(torch.argmax(cosine_eval, dim=-1).cpu().numpy())
                val_targets.extend(targets.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        current_acc = accuracy_score(val_targets, val_preds)
        current_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)

        epoch_history.append({"epoch": epoch + 1, "train_loss": avg_train_loss, "val_loss": avg_val_loss, "val_acc": current_acc, "val_f1": current_f1})
        print(f"Fold {fold} | Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Acc: {current_acc:.4f} | F1: {current_f1:.4f}")

        if current_f1 > best_fold_f1:
            best_fold_f1, best_fold_preds, best_fold_targets = current_f1, val_preds, val_targets
            torch.save({'model_state': model.state_dict(), 'criterion_state': criterion.state_dict()}, checkpoint_path)

    pd.DataFrame(epoch_history).to_csv(history_path, index=False)
    oof_preds[val_idx] = best_fold_preds
    fold_f1_scores.append(best_fold_f1)

    del model, criterion, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

